**Description**

This notebook downloads and processes reverse Liquidity Bootstrapping Pool (rLBP) data from both Balancer V1 and V2 protocols. It handles data cleaning, deduplication, balance conversion, and formatting to prepare datasets for further analysis and modeling.

# Imports

In [ ]:
import pandas as pd
from dune_client.client import DuneClient
from dune_client.query import QueryBase

# Balancer V2

## Removing duplicates and formatting features

In [37]:
balancer_v2_rLBPs = pd.read_csv("../../../media/dune_web_downloads/rLBPs/rLBPs_BalancerV2_Ethereum_pools_addresses.csv", dtype={'poolId': str, 'project_token_address': str})
balancer_v2_rLBPs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   pool_address                       12 non-null     object 
 1   poolId                             12 non-null     object 
 2   pool_type                          12 non-null     object 
 3   collateral_token                   12 non-null     object 
 4   collateral_symbol                  12 non-null     object 
 5   collateral_start_weight            12 non-null     float64
 6   collateral_end_weight              12 non-null     int64  
 7   collateral_initial_balance_raw     12 non-null     float64
 8   project_token_address              12 non-null     object 
 9   project_token_symbol               12 non-null     object 
 10  project_token_initial_balance_raw  12 non-null     float64
 11  start_block_approx                 12 non-null     int64  
 

In [38]:
V2_POOLS_FILE = '../../../media/dune_web_downloads/rLBPs/dev_rLBPs_BalancerV2_cleaned_pools_addresses.csv'

In [39]:
# Known Decimals map
DECIMALS = {
    'USDC': 6,
    'USDT': 6,
    'DAI': 18,
    'WETH': 18,
    'BAL': 18,
    'MIM': 18,
    'FRAX': 18,
    'LUSD': 18,
    'DEFAULT': 18 
}


print("Loading raw data...")
# Read CSV (Ensure poolId is string to prevent truncation)

# Normalize IDs
balancer_v2_rLBPs['poolId'] = balancer_v2_rLBPs['poolId'].str.lower()
balancer_v2_rLBPs['pool_address'] = balancer_v2_rLBPs['pool_address'].str.lower()

print(f"Original Row Count: {len(balancer_v2_rLBPs)}")

# --- STEP 1: IDENTIFY MAIN SALE (Deduplication) ---
# We define the "Main Sale" as the event with the largest change in weights.
# Small changes (e.g. 50 -> 49.5) are usually pauses/reschedules.
# Large changes (e.g. 80 -> 20) are the actual LBP.

balancer_v2_rLBPs['weight_diff'] = abs(balancer_v2_rLBPs['collateral_start_weight'] - balancer_v2_rLBPs['collateral_end_weight'])

# Sort by Pool and Weight Difference (Largest first)
df_sorted = balancer_v2_rLBPs.sort_values(by=['pool_address', 'weight_diff'], ascending=[True, False])

# Keep only the top row for each pool
df_clean = df_sorted.drop_duplicates(subset='pool_address', keep='first').copy()

print(f"Cleaned Row Count: {len(df_clean)} (Removed {len(balancer_v2_rLBPs) - len(df_clean)} duplicate schedules)")

# --- STEP 2: CONVERT RAW BALANCES ---
# Helper to apply decimal conversion
def convert_balance(row, col_name, is_collateral=False):
    raw_val = row[col_name]
    
    # Handle empty/NaN
    if pd.isna(raw_val) or raw_val == '':
        return 0.0
        
    # Determine decimals
    if is_collateral:
        symbol = row.get('collateral_symbol', 'DEFAULT')
        decimals = DECIMALS.get(symbol, 18)
    else:
        # For project tokens, we assume 18 usually, unless we query a token list
        decimals = 18 
        
    try:
        # Convert scientific notation string (e.g., "4e+23") to float
        val = float(raw_val)
        return val / (10 ** decimals)
    except (ValueError, TypeError):
        return 0.0

# Apply conversion
df_clean['collateral_balance'] = df_clean.apply(
    lambda x: convert_balance(x, 'collateral_initial_balance_raw', is_collateral=True), axis=1
)

df_clean['project_balance'] = df_clean.apply(
    lambda x: convert_balance(x, 'project_token_initial_balance_raw', is_collateral=False), axis=1
)

# --- STEP 3: FORMAT DATES ---
df_clean['start_date'] = pd.to_datetime(df_clean['startTime'], unit='s')
df_clean['end_date'] = pd.to_datetime(df_clean['endTime'], unit='s')

# Calculate Duration (in Days)
df_clean['duration_days'] = (df_clean['endTime'] - df_clean['startTime']) / 86400

df_clean = df_clean[df_clean['duration_days'] > 0]

# --- STEP 4: FINAL CLEANUP ---
# Select and reorder useful columns
final_cols = [
    'pool_address', 'poolId', 'collateral_symbol', 
    'collateral_start_weight', 'collateral_end_weight', 
    'collateral_balance', 'project_balance',
    'start_block_approx', 'end_block_est', 
    'start_date', 'end_date', 'duration_days',
    'project_token_address'
]

df_final = df_clean[final_cols]

# Save
df_final.to_csv(V2_POOLS_FILE, index=False)
print(f"\nSuccess! Saved cleaned data to {V2_POOLS_FILE}")
print(df_final[['collateral_symbol', 'collateral_balance', 'project_balance']].head())

Loading raw data...
Original Row Count: 12
Cleaned Row Count: 12 (Removed 0 duplicate schedules)

Success! Saved cleaned data to ../../../media/dune_web_downloads/rLBPs/dev_rLBPs_BalancerV2_cleaned_pools_addresses.csv
  collateral_symbol  collateral_balance  project_balance
7               DAI            400000.0         120000.0
2               DAI          13300000.0        6450000.0
0               DAI          15651134.0        7320000.0
6               DAI           2940000.0        1450000.0
1               DAI          13283548.0        6350000.0


# Balancer V1

In [41]:
# Output Filenames
V1_TRADES_FILE = "../../../media/dune_web_downloads/rLBPs/rLBPs_BalancerV1_trades.csv"
V1_MARKET_FILE = "../../../media/dune_web_downloads/rLBPs/rLBPs_BalancerV1_IDLE_ICHI_market_value.csv"

V1_POOLS_FILE = '../../../media/dune_web_downloads/rLBPs/dev_rLBPs_BalancerV1_cleaned_pools_addresses.csv'

In [42]:
balancer_v1_rLBPs = pd.read_csv("../../../media/dune_web_downloads/rLBPs/rLBPs_BalancerV1_Ethereum_pools_addresses.csv")
balancer_v1_rLBPs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   pool_address                       5 non-null      object
 1   poolId                             5 non-null      object
 2   collateral_token                   5 non-null      object
 3   collateral_symbol                  5 non-null      object
 4   collateral_start_weight            5 non-null      int64 
 5   collateral_end_weight              5 non-null      int64 
 6   collateral_initial_balance_raw     5 non-null      object
 7   project_token_address              5 non-null      object
 8   project_token_symbol               5 non-null      object
 9   project_token_initial_balance_raw  5 non-null      object
 10  start_block_approx                 5 non-null      int64 
 11  end_block_est                      5 non-null      int64 
 12  startTime   

## Removing duplicates and formatting features

In [43]:
# Keep only ICHI and IDLE
balancer_v1_rLBPs = balancer_v1_rLBPs[balancer_v1_rLBPs["project_token_symbol"].isin(['ICHI','IDLE'])]
balancer_v1_rLBPs

,pool_address,poolId,collateral_token,collateral_symbol,collateral_start_weight,collateral_end_weight,collateral_initial_balance_raw,project_token_address,project_token_symbol,project_token_initial_balance_raw,start_block_approx,end_block_est,startTime,endTime
0,0x46935b2489d1468a580ccc3ccba11d1eb7737199,0x46935b2489d1468a580ccc3ccba11d1eb7737199,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,96,50,191079777653152113176,0x903bef1736cddf2a537176cf3c64579c3867a881,ICHI,1470404784010,12265725,12465735,2021-04-18 18:31:54.000 UTC,2021-05-19 15:47:49.000 UTC
2,0xa4a8a79295fa8d0da16b557a9dcad290ff3da321,0xa4a8a79295fa8d0da16b557a9dcad290ff3da321,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,97,5,129611316751214111476,0x875773784af8135ea0ef43b5a374aad105c5d39e,IDLE,17964727403041767662425,11594720,11700000,2021-01-05 13:15:46.000 UTC,2021-01-21 16:50:27.000 UTC


In [44]:
# Known Decimals map
# Updated with V1 specific tokens
DECIMALS = {
    'USDC': 6,
    'USDT': 6,
    'DAI': 18,
    'WETH': 18,
    'BAL': 18,
    'MIM': 18,
    'FRAX': 18,
    'LUSD': 18,
    'ICHI': 9,   # ICHI v1 has 9 decimals
    'IDLE': 18,
}

# Normalize IDs
balancer_v1_rLBPs['poolId'] = balancer_v1_rLBPs['poolId'].str.lower()
balancer_v1_rLBPs['pool_address'] = balancer_v1_rLBPs['pool_address'].str.lower()

# --- STEP 1: DEDUPLICATION ---
balancer_v1_rLBPs['weight_diff'] = abs(balancer_v1_rLBPs['collateral_start_weight'] - balancer_v1_rLBPs['collateral_end_weight'])
df_sorted = balancer_v1_rLBPs.sort_values(by=['pool_address', 'weight_diff'], ascending=[True, False])
df_clean = df_sorted.drop_duplicates(subset='pool_address', keep='first').copy()

# --- STEP 2: CONVERT RAW BALANCES ---
def convert_balance(row, col_name, is_collateral=False):
    raw_val = row[col_name]
    if pd.isna(raw_val) or raw_val == '':
        return 0.0
        
    if is_collateral:
        symbol = row.get('collateral_symbol', 'DEFAULT')
    else:
        symbol = row.get('project_token_symbol', 'DEFAULT')
    
    decimals = DECIMALS.get(symbol, 18)
        
    try:
        val = float(raw_val)
        return val / (10 ** decimals)
    except (ValueError, TypeError):
        return 0.0

df_clean['collateral_balance'] = df_clean.apply(
    lambda x: convert_balance(x, 'collateral_initial_balance_raw', is_collateral=True), axis=1
)

df_clean['project_balance'] = df_clean.apply(
    lambda x: convert_balance(x, 'project_token_initial_balance_raw', is_collateral=False), axis=1
)

# --- STEP 3: FORMAT DATES ---
df_clean['start_date_dt'] = pd.to_datetime(df_clean['startTime'])
df_clean['end_date_dt'] = pd.to_datetime(df_clean['endTime'])

# 1. Calculate Duration
df_clean['duration_days'] = (df_clean['end_date_dt'] - df_clean['start_date_dt']).dt.total_seconds() / 86400

# 2. Format Date Strings to match your V2 file (remove +00:00 timezone info)
# Your V2 file uses "2022-08-31 19:00:00", so we strip timezone
df_clean['start_date'] = df_clean['start_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_clean['end_date'] = df_clean['end_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')

df_clean = df_clean[df_clean['duration_days'] > 0]

# --- STEP 4: FINAL CLEANUP (STRICT SCHEMA MATCH) ---
# I have REMOVED 'project_token_symbol' to match your existing CSV structure
final_cols = [
    'pool_address', 
    'poolId', 
    'collateral_symbol', 
    'collateral_start_weight', 
    'collateral_end_weight', 
    'collateral_balance', 
    'project_balance',          # <--- Previously this was shifted by 'project_token_symbol'
    'start_block_approx', 
    'end_block_est', 
    'start_date', 
    'end_date', 
    'duration_days',
    'project_token_address'
]

df_final = df_clean[final_cols]

# Save
df_final.to_csv(V1_POOLS_FILE, index=False)
print(f"\nSuccess! Saved cleaned data to {V1_POOLS_FILE}")
print("\nCheck alignment below (Should be numeric, numeric, block):")
print(df_final[['collateral_balance', 'project_balance', 'start_block_approx']].head())


Success! Saved cleaned data to ../../../media/dune_web_downloads/rLBPs/dev_rLBPs_BalancerV1_cleaned_pools_addresses.csv

Check alignment below (Should be numeric, numeric, block):
   collateral_balance  project_balance  start_block_approx
0          191.079778      1470.404784            12265725
2          129.611317     17964.727403            11594720


# Join V1 and V2 files

In [45]:
V2_TRADES_FILE = '../../../media/dune_web_downloads/rLBPs/rLBPs_BalancerV2_trades.csv'
V2_MARKET_FILE = '../../../media/dune_web_downloads/rLBPs/rLBPs_BalancerV2_TEMPLE_REMIO_market_value.csv'

# The V1 files were already declareted

In [46]:
v1_pools_file_df = pd.read_csv(V1_POOLS_FILE)
v1_trades_file_df = pd.read_csv(V1_TRADES_FILE)
v1_market_file_df = pd.read_csv(V1_MARKET_FILE)

v2_pools_file_df = pd.read_csv(V2_POOLS_FILE)
v2_trades_file_df = pd.read_csv(V2_TRADES_FILE)
v2_market_file_df = pd.read_csv(V2_MARKET_FILE)

Check if they have the same columns

In [47]:
print(set(v1_pools_file_df.columns) == set(v2_pools_file_df.columns))
print(set(v1_trades_file_df.columns) == set(v2_trades_file_df.columns))
print(set(v1_market_file_df.columns) == set(v2_market_file_df.columns))

True
True
True


## Concatenate dataframes

In [48]:
V1V2_POOLS_FILE = "../../../media/dune_web_downloads/rLBPs/dev_rLBPs_Final_cleaned_pools_addresses.csv"
V1V2_TRADES_FILE = "../../../media/dune_web_downloads/rLBPs/dev_rLBPs_Final_trades.csv"
V1V2_MARKET_FILE = "../../../media/dune_web_downloads/rLBPs/dev_rLBPs_Final_market_values.csv"

In [49]:
pd.concat([v1_pools_file_df,v2_pools_file_df]).to_csv(V1V2_POOLS_FILE, index=False)
pd.concat([v1_trades_file_df, v2_trades_file_df]).to_csv(V1V2_TRADES_FILE, index=False)
pd.concat([v1_market_file_df, v2_market_file_df]).to_csv(V1V2_MARKET_FILE, index=False)